# Statystyka dla Data Analyst/Engineer — Moduł 8: Testy t

W Module 7 zbudowaliśmy cały aparat testowania hipotez na przykładzie proporcji. Teraz
zastosujemy go do najczęściej spotykanego pytania w praktyce: **czy średnie różnią się od
siebie?**. Rodzina **testów t** obsługuje trzy warianty tego pytania: jedna próbka kontra
wartość odniesienia, dwie niezależne grupy, oraz dane sparowane (ten sam obiekt zmierzony
dwa razy).

## Spis treści
1. [Test t dla jednej próbki](#sec1)
2. [Test t dla dwóch prób niezależnych](#sec2)
3. [Sprawdzanie założeń: normalność i równość wariancji](#sec3)
4. [Test t dla prób sparowanych](#sec4)
5. [Wielkość efektu: d Cohena](#sec5)
6. [Podsumowanie i ćwiczenia](#sec6)


Każdy moduł tego kursu zaczyna się od wygenerowania tego samego zestawu danych syntetycznych -- dzięki ustalonemu ziarnu losowości (`np.random.default_rng(42)`) liczby zawsze wyjdą identyczne, więc możesz otworzyć dowolny moduł niezależnie od pozostałych, nawet po restarcie kernela.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

miasto = rng.choice(
    ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["Polnoc", "Poludnie", "Wschod", "Zachod"], size=n)
wiek = rng.integers(18, 70, size=n)

# wydatki miesieczne: rozklad prawoskosny (typowy dla danych o wydatkach)
wydatki_miesieczne = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satysfakcja skorelowana z wydatkami + efekt regionu (celowo rozny) + szum losowy,
# przyciete do skali 1-10 -- dzieki temu Modul 10 (ANOVA) ma realna roznice miedzy grupami
efekt_regionu = {"Polnoc": 0.0, "Poludnie": -0.5, "Wschod": 0.3, "Zachod": 0.8}
satysfakcja = np.clip(
    wydatki_miesieczne / 45
    + np.array([efekt_regionu[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# grupa kampanii A/B -- z celowo wprowadzonym efektem w konwersji (grupa B lepsza)
grupa = rng.choice(["A", "B"], size=n)
prawdopodobienstwo_konwersji = np.where(grupa == "A", 0.10, 0.20)
konwersja = rng.binomial(1, prawdopodobienstwo_konwersji)

# wydatki przed/po wprowadzeniu programu lojalnosciowego (dane sparowane)
wydatki_przed = rng.normal(220, 40, size=n).round(2)
efekt_programu = rng.normal(15, 10, size=n)
wydatki_po = (wydatki_przed + efekt_programu).round(2)

klienci = pd.DataFrame({
    "klient_id": np.arange(1, n + 1),
    "miasto": miasto,
    "region": region,
    "wiek": wiek,
    "wydatki_miesieczne": wydatki_miesieczne,
    "satysfakcja": satysfakcja,
    "grupa_kampanii": grupa,
    "konwersja": konwersja,
    "wydatki_przed": wydatki_przed,
    "wydatki_po": wydatki_po,
})
klienci.head()


<a id="sec1"></a>
## 1. Test t dla jednej próbki

**Jednopróbkowy test t** (*one-sample t-test*) sprawdza, czy średnia próbki różni się od
konkretnej wartości odniesienia. To dokładnie ten sam pomysł co test proporcji z Modułu 7,
tylko dla średniej zamiast odsetka. Sprawdźmy: czy średnie wydatki miesięczne klientów
różnią się od `200` zł?

- H0: `mu = 200`
- H1: `mu != 200`

In [ ]:
from scipy import stats

statystyka_t, wartosc_p = stats.ttest_1samp(klienci["wydatki_miesieczne"], popmean=200)
print(f"t = {statystyka_t:.3f}")
print(f"wartosc p = {wartosc_p:.4f}")

alpha = 0.05
print("Odrzucamy H0" if wartosc_p <= alpha else "Nie odrzucamy H0")


> 📊 **t zamiast z, bo sigma populacji jest nieznana**
>
> `ttest_1samp` używa rozkładu t (a nie z) z dokładnie tego samego powodu co przedziały ufności w Module 6 — nie znamy prawdziwego odchylenia standardowego populacji, więc szacujemy je z próbki, co wprowadza dodatkową niepewność.

<a id="sec2"></a>
## 2. Test t dla dwóch prób niezależnych

**Dwupróbkowy test t** (*independent/two-sample t-test*) porównuje średnie DWÓCH
niezależnych grup. Sprawdźmy: czy średnie wydatki miesięczne różnią się między grupą A i
grupą B naszej kampanii?

- H0: `mu_A = mu_B`
- H1: `mu_A != mu_B`

In [ ]:
wydatki_a = klienci[klienci["grupa_kampanii"] == "A"]["wydatki_miesieczne"]
wydatki_b = klienci[klienci["grupa_kampanii"] == "B"]["wydatki_miesieczne"]

statystyka_t2, wartosc_p2 = stats.ttest_ind(wydatki_a, wydatki_b)
print(f"Srednia A: {wydatki_a.mean():.2f}, Srednia B: {wydatki_b.mean():.2f}")
print(f"t = {statystyka_t2:.3f}, wartosc p = {wartosc_p2:.4f}")


Wynik nie powinien Cię zaskoczyć: przypomnij sobie, że grupa kampanii wpływa w naszych
danych na `konwersja`, a NIE na `wydatki_miesieczne` — spodziewaj się więc dużej wartości
p (brak podstaw do odrzucenia H0). To ważna lekcja: **istotny efekt na jednej zmiennej
(konwersja) nie oznacza istotnego efektu na innej (wydatki)**.

<a id="sec3"></a>
## 3. Sprawdzanie założeń: normalność i równość wariancji

Test t (w klasycznej postaci) zakłada w przybliżeniu normalny rozkład danych w każdej
grupie (albo wystarczająco dużą próbkę, żeby zadziałało CTG z Modułu 5) oraz — w wersji
"studenckiej" — równe wariancje w obu grupach. Dwa szybkie sprawdzenia:

- **Shapiro-Wilk** (`stats.shapiro`) — test normalności. H0: dane pochodzą z rozkładu
  normalnego.
- **Levene** (`stats.levene`) — test równości wariancji. H0: wariancje są równe.

In [ ]:
# normalnosc w kazdej grupie osobno
print("Shapiro-Wilk (grupa A):", stats.shapiro(wydatki_a))
print("Shapiro-Wilk (grupa B):", stats.shapiro(wydatki_b))

# rownosc wariancji
print("Levene:", stats.levene(wydatki_a, wydatki_b))


> ⚠️ **Pamiętasz prawoskośne wydatki z Modułu 2? To tu ma znaczenie**
>
> `wydatki_miesieczne` są wyraźnie prawoskośne (Moduł 2), więc test Shapiro-Wilka najpewniej odrzuci normalność. Czy to problem? Niekoniecznie — dzięki CTG (Moduł 5) **rozkład ŚREDNIEJ próbkowej** i tak zbliża się do normalnego przy naszym `n` (setki obserwacji na grupę), nawet jeśli surowe dane nim nie są. Test t jest dość odporny (*robust*) na umiarkowane odstępstwa od normalności przy dużych próbkach — to bardziej problem dla bardzo małych prób (`n < 15-20`).

Domyślnie `scipy.stats.ttest_ind` zakłada RÓWNE wariancje (klasyczny test Studenta).
Gdy Levene wskazuje na nierówne wariancje, lepszym wyborem jest **test Welcha**
(`equal_var=False`) — nie zakłada równości wariancji i jest bezpiecznym domyślnym wyborem,
gdy nie masz pewności:

In [ ]:
statystyka_welch, p_welch = stats.ttest_ind(wydatki_a, wydatki_b, equal_var=False)
print(f"Test Welcha: t = {statystyka_welch:.3f}, wartosc p = {p_welch:.4f}")


> 📊 **Gdy wątpisz, użyj testu Welcha**
>
> Wielu statystyków rekomenduje domyślne używanie `equal_var=False` niemal zawsze — gdy wariancje faktycznie są równe, test Welcha daje niemal identyczny wynik jak klasyczny test t, a gdy nie są równe, test Welcha jest znacznie bezpieczniejszy. Koszt "ostrożności" jest bardzo niski.

<a id="sec4"></a>
## 4. Test t dla prób sparowanych

**Test t dla prób sparowanych** (*paired t-test*) porównuje DWA pomiary NA TYCH SAMYCH
obiektach (np. przed/po interwencji) — zamiast dwóch niezależnych grup różnych osób. Nasze
kolumny `wydatki_przed`/`wydatki_po` (wydatki tych samych 400 klientów przed i po
wprowadzeniu programu lojalnościowego) to podręcznikowy przykład danych sparowanych.

In [ ]:
statystyka_par, p_par = stats.ttest_rel(klienci["wydatki_po"], klienci["wydatki_przed"])
print(f"Srednia przed: {klienci['wydatki_przed'].mean():.2f}")
print(f"Srednia po: {klienci['wydatki_po'].mean():.2f}")
print(f"t = {statystyka_par:.3f}, wartosc p = {p_par:.6f}")


> ⚠️ **Nie myl testu sparowanego z niezależnym!**
>
> Gdyby ktoś (błędnie) użył `ttest_ind(wydatki_przed, wydatki_po)` zamiast `ttest_rel`, straciłby informację o tym, że to TE SAME osoby mierzone dwukrotnie — test niezależny ignoruje parowanie i jest przez to mniej czuły (traci moc). Zasada: jeśli dwa pomiary dotyczą tych samych jednostek, prawie zawsze chcesz test sparowany.

In [ ]:
# test sparowany to w gruncie rzeczy jednopróbkowy test t na ROZNICACH przed/po
roznice = klienci["wydatki_po"] - klienci["wydatki_przed"]
statystyka_alt, p_alt = stats.ttest_1samp(roznice, popmean=0)
print(f"t (na roznicach) = {statystyka_alt:.3f}, wartosc p = {p_alt:.6f}")


Ten sam wynik co `ttest_rel` — bo to dokładnie to samo pytanie: "czy średnia różnica
`po - przed` różni się istotnie od zera?".

<a id="sec5"></a>
## 5. Wielkość efektu: d Cohena

Wartość p mówi, czy efekt jest **statystycznie istotny**, ale NIE mówi, czy jest **duży**
czy mały praktycznie (przypomnij sobie ostrzeżenie z Modułu 7!). **d Cohena** to popularna
miara wielkości efektu dla różnicy średnich — różnica średnich wyrażona w jednostkach
odchylenia standardowego:

```
d = (srednia1 - srednia2) / odchylenie_std_polaczone
```

Umowne progi interpretacji (Cohen, 1988): `0.2` = mały efekt, `0.5` = średni, `0.8` = duży
— ale to tylko konwencja, kontekst zawsze ma znaczenie.

In [ ]:
def d_cohena(grupa1, grupa2):
    n1, n2 = len(grupa1), len(grupa2)
    s1, s2 = grupa1.std(ddof=1), grupa2.std(ddof=1)
    s_polaczone = (((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2)) ** 0.5
    return (grupa1.mean() - grupa2.mean()) / s_polaczone


d = d_cohena(klienci["wydatki_po"], klienci["wydatki_przed"])
print(f"d Cohena (po vs przed): {d:.3f}")


Zwróć uwagę, że test sparowany z sekcji 4 dał BARDZO małą wartość p (silnie istotny
statystycznie efekt) — sprawdź, czy `d` faktycznie wskazuje na duży efekt, czy tylko na
to, że mieliśmy dużo danych i mały, ale bardzo konsekwentny wzrost.

<a id="sec6"></a>
## 6. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- test t dla jednej próby (`ttest_1samp`) — porównanie średniej z wartością odniesienia,
- test t dla dwóch prób niezależnych (`ttest_ind`) — i różnicę między testem Studenta a
  Welcha,
- sprawdzanie założeń: normalność (Shapiro-Wilk) i równość wariancji (Levene),
- test t dla prób sparowanych (`ttest_rel`) — i dlaczego parowanie ma znaczenie,
- d Cohena jako miarę wielkości efektu, niezależną od wartości p.

> 📝 **Ćwiczenie 1: Test jednej próby dla wieku**
>
> Sprawdź testem t dla jednej próby, czy średni wiek klientów różni się istotnie od `45` lat. Sformułuj H0/H1 i zinterpretuj wynik przy `alpha=0.05`.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
t_wiek, p_wiek = stats.ttest_1samp(klienci["wiek"], popmean=45)
print(f"t={t_wiek:.3f}, p={p_wiek:.4f}")
print("Odrzucamy H0" if p_wiek <= 0.05 else "Nie odrzucamy H0")
```
</details>

> 📝 **Ćwiczenie 2: Wydatki wg miasta (dwie wybrane)**
>
> Porównaj testem Welcha (`equal_var=False`) średnie `wydatki_miesieczne` między Warszawą a Gdańskiem. Sprawdź też Levene'a dla tych dwóch grup przed testem.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
wydatki_warszawa = klienci[klienci["miasto"] == "Warszawa"]["wydatki_miesieczne"]
wydatki_gdansk = klienci[klienci["miasto"] == "Gdansk"]["wydatki_miesieczne"]

print("Levene:", stats.levene(wydatki_warszawa, wydatki_gdansk))
t_miasta, p_miasta = stats.ttest_ind(wydatki_warszawa, wydatki_gdansk, equal_var=False)
print(f"t={t_miasta:.3f}, p={p_miasta:.4f}")
```
</details>

> 📝 **Ćwiczenie 3: d Cohena dla wydatków wg grupy**
>
> Policz d Cohena dla `wydatki_miesieczne` między grupą A i grupą B (użyj funkcji `d_cohena` z sekcji 5). Porównaj z wynikiem testu t z sekcji 2 — czy mały/duży efekt jest zgodny z tym, czy test wyszedł istotny?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
d_grup = d_cohena(wydatki_a, wydatki_b)
print(f"d Cohena: {d_grup:.3f}")
```

Skoro test t z sekcji 2 nie wykazal istotnosci, spodziewaj sie bardzo malego |d| (bliskiego 0) -- male d i brak istotnosci to spojny obraz tego samego braku prawdziwej roznicy.
</details>

> 📝 **Ćwiczenie 4: Test sparowany na podzbiorze**
>
> Wylosuj podpróbkę 20 klientów (`klienci.sample(n=20, random_state=5)`) i powtórz test sparowany `wydatki_po` vs `wydatki_przed` TYLKO na tej podpróbce. Czy wynik nadal jest istotny statystycznie przy tak małej próbie?

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
podprobka = klienci.sample(n=20, random_state=5)
t_mala, p_mala = stats.ttest_rel(podprobka["wydatki_po"], podprobka["wydatki_przed"])
print(f"t={t_mala:.3f}, p={p_mala:.6f}")
```

Efekt jest na tyle silny i konsekwentny (program lojalnosciowy podnosi wydatki niemal kazdemu klientowi), ze prawdopodobnie pozostanie istotny nawet przy n=20 -- ale wartosc p bedzie zazwyczaj wieksza niz przy pelnej probie n=400 (mniej danych = mniej dowodow, jak w Module 7).
</details>

> 🔥 **Wyzwanie: symulacyjna weryfikacja testu t**
>
> Zweryfikuj test t dla dwóch prób NIEZALEŻNYCH metodą permutacyjną (podobnie jak symulacja p-value w Module 7): połącz `wydatki_a` i `wydatki_b` w jedną pulę, 10 000 razy losowo potasuj tę pulę i podziel na dwie grupy o tych samych licznościach co oryginalne A/B, za każdym razem licząc różnicę średnich. Policz odsetek permutacji, w których |różnica permutowana| >= |różnica oryginalna| -- to Twoja permutacyjna wartość p. Porównaj z wartością p z `ttest_ind` z sekcji 2.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import numpy as np

rng = np.random.default_rng(3)
polaczone = np.concatenate([wydatki_a, wydatki_b])
n_a = len(wydatki_a)
roznica_oryginalna = abs(wydatki_a.mean() - wydatki_b.mean())

liczba_permutacji = 10_000
liczba_skrajnych = 0
for _ in range(liczba_permutacji):
    potasowane = rng.permutation(polaczone)
    grupa_a_sym = potasowane[:n_a]
    grupa_b_sym = potasowane[n_a:]
    roznica_sym = abs(grupa_a_sym.mean() - grupa_b_sym.mean())
    if roznica_sym >= roznica_oryginalna:
        liczba_skrajnych += 1

p_permutacyjne = liczba_skrajnych / liczba_permutacji
print(f"Wartosc p (permutacyjna): {p_permutacyjne:.4f}")
print(f"Wartosc p (ttest_ind, sekcja 2): {wartosc_p2:.4f}")
```

Metoda permutacyjna nie zaklada NICZEGO o ksztalcie rozkladu (w przeciwienstwie do testu t) -- to, ze daje bardzo podobny wynik, jest dodatkowym potwierdzeniem, ze test t dziala tu poprawnie mimo skosnosci danych.
</details>

## Co dalej?

Testy t porównują ŚREDNIE zmiennych ciągłych. W **Module 9** zajmiemy się pytaniami o
zmienne **kategoryczne**: czy dwie kategoryczne cechy są od siebie zależne (test
niezależności chi-kwadrat) i czy obserwowany rozkład kategorii pasuje do oczekiwanego
(test zgodności chi-kwadrat).